In [23]:
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from langchain_community.retrievers import BM25Retriever
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import re
import pickle
from langchain_openai import OpenAIEmbeddings
import faiss

In [2]:
meta = pd.read_json("../data/raw/meta_Toys_and_Games.jsonl", lines = True, nrows=10000)
review = pd.read_json("../data/raw/Toys_and_Games.jsonl", lines = True, nrows=10000)

<div class="alert alert-info">
    
### Step 1: Data Exploration and Preprocessing
</div>

In [3]:
review.dtypes

rating                        int64
title                           str
text                            str
images                       object
asin                            str
parent_asin                     str
user_id                         str
timestamp            datetime64[ms]
helpful_vote                  int64
verified_purchase              bool
dtype: object

In [4]:
meta.dtypes

main_category          str
title                  str
average_rating     float64
rating_number        int64
features            object
description         object
price              float64
images              object
videos              object
store                  str
categories          object
details             object
parent_asin            str
bought_together    float64
subtitle               str
author              object
dtype: object

In [5]:
cleaned_meta = meta.drop(columns = ['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'])
cleaned_meta.head()

,main_category,title,average_rating,rating_number,features,description,store,categories,details,parent_asin
0,Toys & Games,"KUNGOON Happy Anniversary Balloon Banner,Weddi...",4.5,241,[],[],Kunggo,[],{'Package Dimensions': '10.12 x 8.03 x 0.51 in...,B08GPM7CQN
1,Toys & Games,Gothic Mothman Plushie Doll with Bright Red Ey...,1.3,2,[🦋 Mothman’s bright red eyes could stare you d...,[🦋 Description: Mothman’s bright red eyes coul...,Felicy,"[Toys & Games, Stuffed Animals & Plush Toys, P...","{'Item Weight': '2.47 ounces', 'Manufacturer r...",B09X9XW42H
2,Toys & Games,Melody Jane Dollhouse Builders DIY 1:24 Scale ...,4.2,67,[1:24 Scale - Plastic - Approximate cut out si...,[],Melody Jane Dolls Houses,"[Toys & Games, Dolls & Accessories, Dollhouse ...","{'Item Weight': '0.48 ounces', 'Manufacturer r...",B01I9QET6M
3,Toys & Games,Traxxas Stampede 4X4: 1/10 Scale 4wd Monster T...,4.5,48,[Waterproof electronics for all-weather drivin...,[Stampede 4X4 is built Traxxas Tough to withst...,Traxxas,"[Toys & Games, Remote & App Controlled Vehicle...",{'Product Dimensions': '15.63 x 13.39 x 8.94 i...,B019XEEX1A
4,Toys & Games,Hot Wheels Monster Truck 1:24 Scale 2022 Bone ...,4.8,17699,[Designed in 1:24 scale with durable die-cast ...,[The Hot Wheels Monster Trucks 1:24 scale die-...,Hot Wheels,"[Toys & Games, Preschool, Pre-Kindergarten Toys]",{'Product Dimensions': '5 x 6.27 x 5.5 inches'...,B09G7K3JWQ


In [6]:
reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns = ['images', 'timestamp', 'user_id', 'verified_purchase'])
cleaned_reviews.head()

,rating,title,text,asin,parent_asin,helpful_vote
0,5,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,B09QH7QJS7,0
1,3,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,B06XYKSKQP,1
2,5,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07SFF3YQW,B07XRSD5R9,0
3,5,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,B007JWWUDW,0
4,3,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,B00MZG6OO8,2


In [7]:
parent_asin = list(pd.Series(np.intersect1d(cleaned_reviews['parent_asin'], cleaned_meta['parent_asin'])))

cleaned_reviews = cleaned_reviews[cleaned_reviews['parent_asin'].isin(parent_asin)]
cleaned_meta = cleaned_meta[cleaned_meta['parent_asin'].isin(parent_asin)]

In [9]:
cleaned_meta['description'] = cleaned_meta['description'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
)
cleaned_meta['description']

128     WizKids Deep Cuts come with highly-detailed fi...
192                                                      
213     Mighty missions call for a mighty team vehicle...
318                                                      
408                                                      
                              ...                        
9450    Perfection in the palm of your hand, Palm Pals...
9688                                                     
9795    Visual perceptions, dexterity and creative thi...
9925                                                     
9995                                                     
Name: description, Length: 144, dtype: str

In [10]:
cleaned_meta['details'] = cleaned_meta['details'].apply(
    lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
)
cleaned_meta['details']

128     Product Dimensions 5 x 6.5 x 1.5 inches Item W...
192     Product Dimensions 7.09 x 5.91 x 3.15 inches I...
213     Product Dimensions 22 x 8 x 16 inches Item Wei...
318     Product Dimensions 12.4 x 6.9 x 6.9 inches Ite...
408     Package Dimensions 16 x 13.06 x 4 inches Item ...
                              ...                        
9450    Product Dimensions 4 x 4 x 5 inches Item Weigh...
9688    Package Dimensions 11.06 x 7.83 x 0.87 inches ...
9795    Brand MindWare Age Range (Description) Kid The...
9925    Product Dimensions 12 x 5 x 19 inches Item Wei...
9995    Fill Material Polyester Pillow Type Bed Pillow...
Name: details, Length: 144, dtype: str

In [14]:
cleaned_meta['features'] = cleaned_meta['features'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)
cleaned_meta['features']

128     Features characters, monsters, and scenery Lit...
192     cotton Main material 100% cotton, allergy & sk...
213     2-IN-1 TRANSFORMING JET: Switch from deluxe je...
318     ►Activity Cube:Includes 12.4x6.9x 6.9 inch woo...
408     ❤Super Value - Eight Cute plush doll in each b...
                              ...                        
9450    5 inches long from top to bottom High quality ...
9688    Polyester 🌟【Kids Chef Hat Apron Set】: Children...
9795    Fun and educational toys for children all ages...
9925    The bricks are Very Tiny, please look over the...
9995    Microfiber,polyester SUPER SOFT & GREAT SUPPOR...
Name: features, Length: 144, dtype: str

In [11]:
cleaned_meta['categories'] = cleaned_meta['categories'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)
cleaned_meta['categories']

128          Toys & Games Games & Accessories Board Games
192     Toys & Games Preschool Pre-Kindergarten Toys S...
213     Toys & Games Preschool Pre-Kindergarten Toys P...
318     Toys & Games Baby & Toddler Toys Early Develop...
408     Toys & Games Stuffed Animals & Plush Toys Plus...
                              ...                        
9450    Toys & Games Preschool Pre-Kindergarten Toys S...
9688    Toys & Games Dress Up & Pretend Play Kitchen T...
9795        Toys & Games Arts & Crafts Craft Kits Mosaics
9925    Toys & Games Preschool Pre-Kindergarten Toys A...
9995    Toys & Games Stuffed Animals & Plush Toys Plus...
Name: categories, Length: 144, dtype: str

##### Description of text preprocessing decisions


<div class="alert alert-info">
    
### Step 2 and 3: BM25 Search and Semantic Search
</div>

In [ ]:
# code adapted from lecture 5

# combining all the important and useful columns from the meta dataset
products = (
    cleaned_meta['title'].fillna('') + ' ' +
    cleaned_meta['description'].fillna('') + ' ' +
    cleaned_meta['features'].fillna('') + ' ' +
    cleaned_meta['categories'].fillna('')
)
products = products.tolist()
queries = cleaned_reviews['title'].tolist()

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", "", text)
    return text.split()

tokenized_products = [simple_tokenize(p) for p in products]
bm25 = BM25Okapi(tokenized_products)

model = SentenceTransformer("all-MiniLM-L6-v2")
product_embeddings = model.encode(products).astype("float32")

# using faiss to index product embeddings for semantic search
index = faiss.IndexFlatL2(product_embeddings.shape[1])
index.add(product_embeddings)

def bm25_search(query, top_k=3):
    tokenized_query = simple_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [(products[i], scores[i]) for i in ranked_idx]

def embedding_search(query, top_k=3):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [(products[i], distances[0][rank]) for rank, i in enumerate(indices[0])]

for q in queries:
    print("=" * 80)
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    for rank, (product, score) in enumerate(bm25_search(q), start=1):
        print(f"{rank}. ({score:.3f}) {product}")

    print("\nEmbedding search top results:")
    for rank, (product, score) in enumerate(embedding_search(q), start=1):
        print(f"{rank}. ({score:.3f}) {product}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: ADORABLE

BM25 top results:
1. (4.382) Big Time Toys Sea Monkeys Ocean Zoo Deluxe Kit Set- Colors May Vary The original, amazing live sea monkeys a classic set to create your very own adorable Aqua-Pets The plastic aquarium is equipped with a ventilated lid, built-in magnifiers and molded seascape bottom Tank measures 5.625" tall ****Please note that results of egg hatched VARY for packages**** Your Sea Monkey pets MAY live up to two years The original, amazing live sea monkeys a classic set to create your very own adorable Aqua-Pets The plastic aquarium is equipped with a ventilated lid, built-in magnifiers and molded seascape bottom Tank measures 5.625" tall Your Sea Monkey pets may live up to two years Toys & Games STEM Toys Science Nature Exploration Toys
2. (4.180) Worlds Smallest Good Luck Trolls. Mini 1 inch Tall Toy Action Figure with an Extra 1.5 inches of Hair! Six Adorable Good Luck Trolls to Collect! Great for School Project, Arts Crafts, Party Favors For over 60 yea

In [25]:
with open("../data/processed/bm25_index.pkl", "wb") as i:
    pickle.dump(bm25, i)

with open("../data/processed/tokenized_products.pkl", "wb") as i:
    pickle.dump(tokenized_products, i)

In [26]:
faiss.write_index(index, "../data/processed/product_index.faiss")

<div class="alert alert-info">
    
### Step 4: Qualitative Evaluation
</div>

<div class="alert alert-info">
    
### Step 5: Web App
</div>